# Lab 2 (simplified) · Part 3 — Turning the model into a small service
### EUROCONTROL AI Lab · Exercise 2S-3

A model is useful in operations only if **other systems can ask it for predictions**. Here we run the model trained in Part 1 as a small **web service**
on **your own computer** (`127.0.0.1` = "this computer": nothing leaves the PC) and talk to it the way any other system would.

```
 this notebook / web browser ── request: one weather observation ──►  serve_simple.py  ──►  lowvis_core.py + artifacts/
                             ◄── answer: {"probability": 0.57, "alert": true, ...} ──
```

| | |
|---|---|
| ⏱ **Time** | about 45 minutes |
| 🎓 **Prerequisites** | run **Part 1** first: it saves the model in the `artifacts` folder |

**What you will learn**
1. What a prediction **service** is, and why the model is loaded once and then answers many requests.
2. How the service **checks** its inputs and refuses nonsense (for example a temperature in °F).
3. Why training and serving must prepare the inputs with **the same code**.

The model predicts visibility < 1 500 m **3 hours later** — a **proxy** for Low-Visibility Procedures, as written in its model card.

> **How to use this notebook**
> - Run the cells **one at a time, from top to bottom**: click a cell and press **Shift + Enter**.
> - Every line of code has a comment (the text after `#`) explaining what it does. You do not need to write any code.
> - After each plot there is a short **📖 Reading** that explains what the plot tells us.
> - If something breaks, use *Kernel → Restart Kernel and Run All Cells*.
> - This notebook is yours to keep: you can re-run it later and change the numbers to experiment.

---
## 0 · Setup

In [1]:
import sys                                        # sys: gives the path of the Python program
import time                                       # time: waiting and measuring durations
import subprocess                                 # subprocess: starts another program (the service) in the background
from pathlib import Path                          # Path: file paths that work on every operating system
import numpy as np                                # numpy: calculations on arrays
import pandas as pd                               # pandas: tables of data
import requests                                   # requests: sends messages (HTTP requests) to the service

LAB_DIR = Path(".").resolve()                     # this lab's folder (where serve_simple.py is)
PORT = 8000                                       # the "door number" the service listens on
URL = "http://127.0.0.1:" + str(PORT)             # the address of the service on this computer

---
## 1 · Start the service
The next cell starts `serve_simple.py` in the background and waits until it answers. *(You can open `serve_simple.py` in JupyterLab: it is short and every line is commented.)*

In [2]:
def service_is_running():                          # True if the service answers
    try:                                           # try to contact it...
        return requests.get(URL + "/health", timeout=1).ok   # ...on its /health address
    except requests.ConnectionError:               # nobody answers...
        return False                               # ...so it is not running


server = None                                      # no service started by this notebook yet
if not service_is_running():                       # if the service is not already running...
    command = [sys.executable, "-m", "uvicorn", "serve_simple:app", "--host", "127.0.0.1", "--port", str(PORT)]   # the command
    server = subprocess.Popen(command, cwd=LAB_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)   # start it
    for attempt in range(60):                      # wait up to 30 seconds...
        if service_is_running():                   # ...until it answers
            break                                  # it answers: stop waiting
        if server.poll() is not None:              # if it stopped by itself...
            print(server.stdout.read())            # ...show its error (often: Part 1 was not run)
            break                                  # stop waiting
        time.sleep(0.5)                            # wait half a second and try again
print("Service running:", service_is_running())    # final check

Service running: True


Open **[http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)** in a new browser tab: the service documents itself — every input, its unit and its allowed range.

---
## 2 · Ask the service
First we check that it is alive and read its **model card** (the description saved in Part 1).

In [3]:
print(requests.get(URL + "/health").json())        # is it alive? which model version?
card = requests.get(URL + "/model").json()         # the model card
print("Model:", card["name"])                      # its name
print("Predicts:", card["task"])                   # what it predicts
print("Note:", card["target_note"])                # what the target really means
print("Results on 2025:", card["test_2025"])       # how well it did on the test year

{'status': 'ok', 'model_version': '1.0.0'}
Model: Low-visibility nowcast LIMC, simplified (LVP proxy)
Predicts: P(visibility < 1500 m exactly 3 hours after the METAR)
Note: Proxy for LVP, derived from METAR visibility only; real LVP depend on RVR, ceiling and local procedures.
Results on 2025: {'precision': 0.26, 'recall': 0.46, 'hits': 63, 'false alarms': 176, 'misses': 75, 'onsets caught': '26 of 78'}


Now we send one observation — a calm, humid winter evening — and read the answer:

In [4]:
winter_evening = {                                 # one observation, as a dictionary:
    "time": "2025-01-06T20:50:00",                 #   6 January 2025, 20:50 UTC
    "visibility_m": 3500,                          #   visibility 3 500 m
    "temperature_c": 1.0,                          #   1 °C
    "dewpoint_c": 0.0,                             #   dew point 0 °C (almost saturated air)
    "humidity_pct": 93,                            #   humidity 93 %
    "wind_speed_kt": 1,                            #   almost no wind
}                                                  # (end of the observation)
answer = requests.post(URL + "/predict", json=winter_evening)   # send it to the /predict address
print("HTTP status:", answer.status_code)          # 200 = OK
print(answer.json())                               # the service's answer

summer_afternoon = {"time": "2025-07-15T14:50:00", "visibility_m": 9999, "temperature_c": 31,
                    "dewpoint_c": 17, "humidity_pct": 43, "wind_speed_kt": 8}   # a hot, dry summer afternoon
print(requests.post(URL + "/predict", json=summer_afternoon).json())             # send it and show the answer

HTTP status: 200
{'probability': 0.55315, 'alert': True, 'threshold': 0.5, 'model_version': '1.0.0'}
{'probability': 0.0001, 'alert': False, 'threshold': 0.5, 'model_version': '1.0.0'}


📖 **Reading:** the winter evening gets a probability above the threshold → **alert**; the summer afternoon gets almost 0. The service answers in a standard format (JSON) that any system can read.

---
## 3 · The service protects the model
What happens if a client sends nonsense? We try three mistakes on purpose.

In [5]:
wrong = dict(winter_evening)                       # copy the good observation...
wrong["temperature_c"] = 34.0                      # ...but with the temperature in °F by mistake (1 °C ≈ 34 °F)... still plausible!
print("Temperature 34 (°F sent as °C):", requests.post(URL + "/predict", json=wrong).json())   # accepted: looks possible

wrong = dict(winter_evening)                       # copy again...
wrong["temperature_c"] = 95.0                      # ...a summer temperature in °F: impossible in °C
answer = requests.post(URL + "/predict", json=wrong)   # send it
print("Temperature 95:", answer.status_code, answer.json()["detail"][0]["msg"])   # refused, with the reason

wrong = dict(winter_evening)                       # copy again...
del wrong["visibility_m"]                          # ...and forget the visibility
answer = requests.post(URL + "/predict", json=wrong)   # send it
print("Missing visibility:", answer.status_code, answer.json()["detail"][0]["msg"])   # refused, with the reason

wrong = dict(winter_evening)                       # copy again...
wrong["humidity_pct"] = 130                        # ...a humidity above 100 %
answer = requests.post(URL + "/predict", json=wrong)   # send it
print("Humidity 130 %:", answer.status_code, answer.json()["detail"][0]["msg"])   # refused, with the reason

Temperature 34 (°F sent as °C): {'probability': 0.17533, 'alert': False, 'threshold': 0.5, 'model_version': '1.0.0'}
Temperature 95: 422 Input should be less than or equal to 50
Missing visibility: 422 Field required
Humidity 130 %: 422 Input should be less than or equal to 100


📖 **Reading:** HTTP **422** means *"your data breaks the rules"*: impossible values and missing fields are refused **before** reaching the model.
But look at the first case: a temperature of 34 is *possible*, so the service accepts it — and the answer changes completely (the alert disappears).
**Checks catch impossible values, not plausible mistakes.**
That is why real services also need **monitoring** of what they receive.

---
## 4 · Does the service give the same answers as the notebook?
We send 200 real hours of 2025 to the service and compare its answers with the probabilities computed directly from the saved model.

In [6]:
sys.path.insert(0, str(LAB_DIR))                   # let Python find lowvis_core.py
import lowvis_core                                 # the shared code

weather = lowvis_core.load_weather(LAB_DIR.parent / "data" / "LIMC.csv")   # the weather reports, in aviation units
sample = weather.loc["2025-01-01":"2025-01-15"].dropna().head(200)          # 200 complete hours of January 2025

service_answers = []                               # the probabilities returned by the service
for moment, row in sample.iterrows():              # for each hour...
    observation = {                                # ...build the observation...
        "time": moment.strftime("%Y-%m-%dT%H:%M:%S"),          # time as text
        "visibility_m": min(float(row["visibility_m"]), 9999),  # visibility (capped at 9999 m)
        "temperature_c": float(row["temperature_c"]),           # temperature
        "dewpoint_c": float(row["dewpoint_c"]),                 # dew point
        "humidity_pct": float(row["humidity_pct"]),             # humidity
        "wind_speed_kt": float(row["wind_speed_kt"]),           # wind speed
    }                                              # (end of the observation)
    reply = requests.post(URL + "/predict", json=observation).json()   # ...ask the service...
    service_answers.append(reply["probability"])   # ...and keep its probability

saved = lowvis_core.SavedModel(LAB_DIR / "artifacts")          # the saved model, used directly
direct = saved.probability(lowvis_core.build_features(sample, sample.index))   # the same 200 predictions, directly
difference = np.abs(np.array(service_answers) - direct).max()  # the largest difference
print("Largest difference service vs notebook:", difference)   # only rounding (the service rounds to 5 decimals)

Largest difference service vs notebook: 5.007972717296916e-06


📖 **Reading:** the answers are identical (apart from rounding) because the service and the notebooks prepare the inputs with **the same file**, `lowvis_core.py`.
If they used two different copies of the preparation code, a small difference — for example a missing unit conversion — would silently change every answer.

---
## 5 · The web form
Open **[http://127.0.0.1:8000/](http://127.0.0.1:8000/)**: a simple page that sends the form to the same `/predict` address.
Try the two example buttons, then describe a situation of your own.

---
## 6 · Stop the service

In [7]:
if server is not None:                             # if this notebook started the service...
    server.terminate()                             # ...ask it to stop...
    server.wait(timeout=10)                        # ...and wait until it has stopped
    print("Service stopped")                       # confirm
else:                                              # otherwise it was started somewhere else...
    print("Stop the service where you started it (Ctrl+C in its terminal)")   # ...stop it there

Service stopped


---
## 7 · Questions for discussion
1. The service refused a temperature of 95 but accepted 34. What else could help catch plausible-but-wrong inputs?
2. Our service answers only on this computer. What would change — technically and in terms of responsibility — if other systems had to use it?
3. The model card says the target is a **proxy** for LVP. Who should read that sentence, and when?